# 05 Invalid Values

Identify values that fall outside expected ranges or violate basic data rules.

## 1. Preparation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/04_data_types.parquet")
df_clean = df.copy()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 95,000
Columns: 33


## 2. Detection

Review numerical ranges and identify potentially invalid values.

In [2]:
numeric_columns = [
    "age",
    "years_experience",
    "tenure_years",
    "performance_review_score",
]

results = []

for column in numeric_columns:
    values = df[column]

    results.append({
        "column": column,
        "min": values.min(),
        "max": values.max(),
        "mean": round(values.mean(), 2),
        "median": values.median(),
        "negative": (values < 0).sum(),
        "zero": (values == 0).sum(),
        "missing": values.isna().sum()
    })

display(pd.DataFrame(results))

,column,min,max,mean,median,negative,zero,missing
0,age,20.0,68.0,35.60,35.0,0,0,2877
1,years_experience,0.0,40.0,14.67,14.0,0,5327,937
2,tenure_years,0.0,16.0,2.91,3.0,0,23388,0
3,performance_review_score,1.0,5.0,3.39,3.4,0,0,14758


### 2.1 Salary Validation

Convert salaries to USD using fixed exchange rates to enable comparisons across currencies.

These rates are predefined assumptions, not historical exchange rates. Original salary amounts and currencies remain unchanged.

In [3]:
exchange_rates = {
    "USD": 1.0,
    "EUR": 0.8962,
    "GBP": 0.7572,
    "KES": 129.74,
    "ZAR": 16.5411,
    "PHP": 62.7680,
    "AUD": 1.4374,
    "INR": 95.6024,
    "CAD": 1.4287,
    "BRL": 4.9870,
    "SGD": 1.2811,
    "NGN": 1327.38
}

rates = df_clean["currency"].map(exchange_rates)

assert rates.notna().all(), "Missing exchange rates."
assert (rates > 0).all()

df_clean["salary_raw_USD"] = (
    df_clean["salary_raw"] / rates
).round(2)

display(
    df_clean[
        ["salary_raw", "currency", "salary_raw_USD"]
    ].sample(20, random_state=42)
)

,salary_raw,currency,salary_raw_USD
43450,3.889600e+04,CAD,27224.75
14791,3.370893e+04,EUR,37613.18
28617,8.253184e+04,AUD,57417.45
60875,2.647962e+04,GBP,34970.44
50419,7.037000e+04,USD,70370.00
79901,5.595130e+04,AUD,38925.35
91111,1.228340e+06,ZAR,74259.87
34188,3.622457e+04,GBP,47840.16
18106,4.407540e+05,ZAR,26645.99
56770,2.637500e+04,USD,26375.00


In [4]:
display(
    df_clean["salary_raw_USD"]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99])
    .round(2)
)

count       95000.00
mean        52252.48
std        180985.69
min           156.26
1%          19446.37
25%         32659.76
50%         42454.72
75%         55516.63
99%        101707.67
max      11022793.00
Name: salary_raw_USD, dtype: float64

### 2.2 Salary Extremes

Inspect the lowest and highest salaries to identify possible data errors.

In [5]:
columns = [
    "employee_id",
    "salary_raw_USD",
    "currency",
    "pay_band",
    "job_level",
    "job_title"
]

print("Lowest 20 salaries:")
display(
    df_clean.nsmallest(20, "salary_raw_USD")[columns]
)

print("Highest 20 salaries:")
display(
    df_clean.nlargest(20, "salary_raw_USD")[columns]
)

Lowest 20 salaries:


,employee_id,salary_raw_USD,currency,pay_band,job_level,job_title
89512,E175308,156.26,PHP,Band 1,IC,Sales Development Rep
25918,LEG-39183,188.28,CAD,Band 1,IC,Marketing Coordinator
13015,E157892,194.06,PHP,Band 1,IC,IT Support Specialist
44469,E149612,196.57,PHP,Band 1,IC,Financial Analyst
34887,E161098,200.64,INR,Band 1,IC,Recruiter
29045,E146298,206.02,GBP,Band 1,IC,UX Designer
10262,E120788,210.74,KES,Band 1,IC,CONTENT STRATEGIST
7896,E190851,232.09,EUR,Band 1,IC,Product Analyst
17221,E108365,245.32,USD,Band 1,IC,UX Designer
23730,LEG-83628,247.04,BRL,Band 1,IC,Marketing Coordinator


Highest 20 salaries:


,employee_id,salary_raw_USD,currency,pay_band,job_level,job_title
698,E136739,11022793.00,USD,Band 4,Manager / Lead,Finance Manager
27060,LEG-82246,10489718.86,NGN,Band 3,Executive,CFO
70168,LEG-63324,9880869.00,KES,Band 4,Manager / Lead,Senior Product Manager
36203,E144403,9363247.49,GBP,Band 4,Manager / Lead,Sales Manager
66431,E103941,9218186.79,EUR,Band 3,Manager / Lead,Senior Product Manager
53786,E108877,8455808.79,USD,Band 3,Senior,Staff Engineer
77112,E144185,8052150.03,GBP,Band 3,Manager / Lead,Design Lead
44487,E133221,7652000.95,AUD,Band 3,Executive,VP PRODUCT
79363,E194252,7562911.87,GBP,Band 3,Executive,VP Design
30609,E118684,7485219.77,BRL,Band 3,Manager / Lead,Finance Manager


### 2.3 Salary Flagging

Salary information should not be modified without verification.

Flag salaries below $1,000 or above $1,000,000 USD for further review.

These thresholds are screening rules, not confirmed data errors. All original values are preserved.

In [6]:
df_clean["salary_needs_review"] = (
    (df_clean["salary_raw_USD"] < 1_000) |
    (df_clean["salary_raw_USD"] > 1_000_000)
)

print(f"Flagged records: {df_clean['salary_needs_review'].sum():,}")
print(f"Unflagged records: {(~df_clean['salary_needs_review']).sum():,}")

display(
    df_clean.loc[
        df_clean["salary_needs_review"],
        [
            "employee_id",
            "salary_raw",
            "currency",
            "salary_raw_USD",
            "pay_band",
            "job_level",
            "salary_needs_review"
        ]
    ].head(20)
)

Flagged records: 234
Unflagged records: 94,766


,employee_id,salary_raw,currency,salary_raw_USD,pay_band,job_level,salary_needs_review
413,E119922,447.75,GBP,591.32,Band 2,IC,True
698,E136739,11022793.00,USD,11022793.00,Band 4,Manager / Lead,True
987,E190369,1608.00,BRL,322.44,Band 1,IC,True
1365,LEG-9937,8140.00,ZAR,492.11,Band 2,IC,True
1447,E110405,2947207.23,USD,2947207.23,Band 1,IC,True
2340,E179316,2148516.14,GBP,2837448.68,Band 1,IC,True
2606,E149328,267.18,EUR,298.13,Band 2,IC,True
2890,E166398,265.00,USD,265.00,Band 1,IC,True
3884,E169071,1015259.00,NGN,764.86,Band 3,Manager / Lead,True
4233,E188234,22730.00,PHP,362.13,Band 2,Manager / Lead,True


In [7]:
print(f"Total records: {len(df_clean):,}")
print(f"Flagged salaries: {df_clean['salary_needs_review'].sum():,}")
print(f"Unflagged salaries: {(~df_clean['salary_needs_review']).sum():,}")

print(f"Below $1,000: {(df_clean['salary_raw_USD'] < 1_000).sum():,}")
print(f"Above $1,000,000: {(df_clean['salary_raw_USD'] > 1_000_000).sum():,}")

assert df_clean["salary_raw"].equals(df["salary_raw"])

Total records: 95,000
Flagged salaries: 234
Unflagged salaries: 94,766
Below $1,000: 113
Above $1,000,000: 121


### 2.4 Other Numeric Values

Check employee age, experience, tenure, and performance scores for invalid values.

In [8]:
invalid_checks = {
    "age": df_clean["age"].notna() & ~df_clean["age"].between(1, 120),
    "years_experience": df_clean["years_experience"].notna() & (df_clean["years_experience"] < 0),
    "tenure_years": df_clean["tenure_years"].notna() & (df_clean["tenure_years"] < 0),
    "performance_review_score": (
        df_clean["performance_review_score"].notna()
        & ~df_clean["performance_review_score"].between(1, 5)
    )
}

results = pd.DataFrame({
    "column": invalid_checks.keys(),
    "invalid_count": [
        mask.fillna(False).sum()
        for mask in invalid_checks.values()
    ]
})

display(results)

,column,invalid_count
0,age,0
1,years_experience,0
2,tenure_years,0
3,performance_review_score,0


## 3. Export

Save the dataset with salary review flags. Original salary values remain unchanged.

In [9]:
from pathlib import Path

assert len(df_clean) == len(df)
assert df_clean["salary_raw"].equals(df["salary_raw"])
assert all(mask.fillna(False).sum() == 0 for mask in invalid_checks.values())
assert df_clean["salary_needs_review"].sum() == 234

output_path = Path("data/processed/05_invalid_values.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, index=False)

saved = pd.read_parquet(output_path)

pd.testing.assert_frame_equal(df_clean, saved)

print(f"Saved: {output_path}")
print(f"Rows: {len(saved):,}")
print(f"Columns: {len(saved.columns)}")
print(f"Salaries flagged: {saved['salary_needs_review'].sum():,}")

Saved: data/processed/05_invalid_values.parquet
Rows: 95,000
Columns: 35
Salaries flagged: 234


## 4. Summary

- Converted salaries from 12 currencies into USD using fixed exchange rates.
- Identified 234 salaries requiring further review.
- Flagged 113 salaries below $1,000.
- Flagged 121 salaries above $1,000,000.
- Found no invalid values in age, experience, tenure, or performance scores.
- Preserved all original salary values.

### Final Dataset

- **95,000** employee records.
- **35** columns.
- All validation checks passed.